# 발표 그래프 다시 그리기 — 원래 V4(취약 후보 43개) 기준

PPT의 12·13·15·16쪽 그래프가 CPU로 다시 학습한 V4(47개) 결과로 그려져 있어서, **원래 V4로 02·03·04를 다시 돌리고 발표용 그래프와 숫자표를 한 번에 만듭니다.**

**하는 일**
1. 원래 V4 CSV로 02(판정·유형) → 03(2단계 모델·과거 평균 비교) → 04(감소 분해)를 차례로 실행합니다. 코드는 재현 코드 노트북을 그대로 불러 씁니다.
2. 취약 후보가 **17·3·14·9 = 43개**인지 확인합니다. 다르면 멈춥니다.
3. 발표용 그래프 7장 + 숫자표(엑셀)를 `results/발표그래프/`에 저장합니다.

| 파일 | 넣을 곳 |
|---|---|
| `G1_평시범위_히스토그램.png` | 12쪽(10 취약 판정) 왼쪽 |
| `G2_3축_예시.png` | 12쪽 오른쪽 |
| `G3_이벤트별_순위상관.png` | 13쪽(11) 왼쪽 |
| `G4_MAE_개선율.png` | 13쪽 오른쪽 |
| `G5_Impact_Cumulative.png` | 15쪽(결과②) |
| `G6_방문_지출.png` (+ `G6b_방문_지출_여름보조.png`) | 16쪽(결과③) 왼쪽 |
| `G7_고객층.png` | 16쪽 오른쪽 |
| `발표숫자.xlsx` | 14·16·13·19쪽 숫자 확인용 |

**준비**
- Drive `big_contest` 아래에 재현 코드 노트북(`02_…`, `03_…`, `04_…`)이나 **`재현코드.zip`을 그대로** 올려 두면 됩니다. 위치는 자동으로 찾고, zip이면 `재현코드_풀림` 폴더에 풀어서 씁니다.
- 03은 기상 파일, 04는 카드 데이터2를 씁니다(재현 코드와 같은 파일).
- 런타임은 CPU로 충분합니다(baseline을 다시 학습하지 않음). 전체 약 5~10분.


In [ ]:
# 0. 설정 — 경로 확인
from pathlib import Path
import io, json, time, contextlib, warnings
ROOT = Path('/content/drive/MyDrive/big_contest')
try:
    from google.colab import drive; drive.mount('/content/drive')
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib', 'lightgbm'])
except ImportError:
    pass
import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt
import koreanize_matplotlib
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

# ① 원래 V4 (43개가 나온 파일, 10/11 저장본). GPU로 01을 다시 돌린 파일이어도 됩니다.
V4_ORIG = ROOT / 'outputs' / '38_최종baseline_V4.csv'
# ② 결과 폴더 (기존 results/유형나누기 등은 건드리지 않음)
OUT = ROOT / 'results' / '발표그래프'; OUT.mkdir(parents=True, exist_ok=True)
# ③ 12쪽 예시 업종·이벤트
EX_G, EX_EP = '강남(서울동남권)|세탁소', '폭염2'
EXPECT = {('강남(서울동남권)', '폭염'): 17, ('강남(서울동남권)', '호우'): 3, ('춘천', '폭염'): 14, ('춘천', '호우'): 9}

# 재현코드 노트북이 압축 파일(재현코드*.zip)로만 있으면 big_contest/재현코드_풀림 폴더에 풀어서 씀
import zipfile
if not list(ROOT.rglob('02_취약판정_유형나누기.ipynb')):
    zips = sorted(ROOT.rglob('재현코드*.zip'), key=lambda p: p.stat().st_mtime)
    assert zips, '재현코드 노트북이나 재현코드.zip을 Drive의 big_contest 아래에 올려 주세요'
    with zipfile.ZipFile(zips[-1]) as zf:
        for info in zf.infolist():
            name = info.filename
            if not info.flag_bits & 0x800:          # 한글 파일 이름 깨짐 방지
                try: name = name.encode('cp437').decode('utf-8')
                except UnicodeError: pass
            if name.endswith('.ipynb'):
                dst = ROOT / '재현코드_풀림' / Path(name).name; dst.parent.mkdir(exist_ok=True)
                dst.write_bytes(zf.read(info))
    print('압축 풀기:', zips[-1], '→', ROOT / '재현코드_풀림')

def find_nb(name):
    hits = sorted(ROOT.rglob(name), key=lambda p: p.stat().st_mtime)
    assert hits, f'{name}을(를) {ROOT} 아래에서 찾지 못했습니다 — 재현 코드 노트북을 Drive에 올려 주세요'
    return hits[-1]
NB02, NB03, NB04 = find_nb('02_취약판정_유형나누기.ipynb'), find_nb('03_2단계모델.ipynb'), find_nb('04_감소분해.ipynb')
assert V4_ORIG.exists(), f'V4_ORIG 경로를 확인하세요: {V4_ORIG}'
print('V4:', V4_ORIG, '| 수정 시각', time.strftime('%Y-%m-%d %H:%M', time.localtime(V4_ORIG.stat().st_mtime)))
print('노트북:', NB02, NB03, NB04, sep='\n  ')

# 발표용 그래프 스타일 (PPT 색에 맞춘 남색·주황 계열)
C = dict(navy='#1f2a5c', blue='#2a78d6', orange='#eb6834', teal='#1baf7a', yellow='#eda100', red='#d9534f',
         gray='#9aa0aa', ink='#16181d', ink2='#555a64', grid='#e6e7eb', shade='#fbe9e4', band='#d7efe9')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': C['ink2'],
                     'axes.labelcolor': C['ink2'], 'xtick.color': C['ink2'], 'ytick.color': C['ink2'],
                     'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': 0.8, 'axes.axisbelow': True,
                     'axes.unicode_minus': False, 'figure.dpi': 110, 'savefig.dpi': 220, 'font.size': 10})
def save(fig, name):
    fig.savefig(OUT / name, bbox_inches='tight', facecolor='white'); plt.show(); print('저장:', OUT / name)


## 1. 02 판정 — 원래 V4로 다시 실행 (43개 확인)

In [ ]:
# 1. 재현 코드 노트북을 불러 실행하는 함수 (경로만 바꾸고 출력은 숨김)
def run_nb(path, repl, stop=None, skip=()):
    nb = json.load(open(path, encoding='utf-8'))
    cells = [''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code']
    ns = {'__name__': '__main__', '_CAP': {}}
    repl = {"ROOT = Path('/content/drive/MyDrive/big_contest')": f"ROOT = Path(r'{ROOT}')", **repl}
    done = {k: 0 for k in repl}
    log = io.StringIO()
    show, backend = plt.show, matplotlib.get_backend()
    plt.show = lambda *a, **k: plt.close('all')
    try:
        with contextlib.redirect_stdout(log):
            for i, src in enumerate(cells):
                if stop and src.startswith(stop): break
                if any(src.startswith(s) for s in skip): continue
                for old, new in repl.items():
                    if old in src: src = src.replace(old, new); done[old] += 1
                ns['display'] = lambda *a, **k: None
                exec(compile(src, f'{path.name}#cell{i}', 'exec'), ns)
                plt.close('all')
    finally:
        plt.show = show
    miss = [k for k, v in done.items() if v == 0]
    assert not miss, f'{path.name}에서 바꿀 줄을 찾지 못했습니다: {miss}'
    return ns, log.getvalue()

t0 = time.time()
ns2, log2 = run_nb(NB02, {
    "V4_PATH = find('38_최종baseline_V4.csv')": f"V4_PATH = Path(r'{V4_ORIG}')",
    "OUT = ROOT/'results'/'유형나누기'": f"OUT = Path(r'{OUT / '02_유형나누기'}')",
    "return tuple(np.percentile(ds / bs, RANGE_Q))": "_CAP[(g, tuple(dates))] = ds / bs; return tuple(np.percentile(ds / bs, RANGE_Q))",
}, stop='# 7.')
TYPE_X = OUT / '02_유형나누기' / '유형나누기_결과.xlsx'
S2, CAND, EPI, EP2, D2 = ns2['S'], ns2['CAND'], ns2['EPI'], ns2['EP'], ns2['D']
cnt = CAND.groupby(['대상', '종류']).size().to_dict()
print(f'02 완료 ({time.time() - t0:.0f}초) — 취약 후보:', cnt, '합계', len(CAND))
assert cnt == EXPECT, '⚠ 43개가 아닙니다. V4_ORIG가 원래 V4(GPU) 파일인지 확인하세요 (CPU로 만든 V4면 47개)'


## 2. 03 과거 평균 비교 · 04 감소 분해

In [ ]:
# 2. 03(2단계 모델·과거 평균 비교), 04(감소 분해)
t0 = time.time()
ns3, log3 = run_nb(NB03, {
    "V4_PATH = find('38_최종baseline_V4.csv')": f"V4_PATH = Path(r'{V4_ORIG}')",
    "TYPE_PATH = find('유형나누기_결과.xlsx')": f"TYPE_PATH = Path(r'{TYPE_X}')",
    "OUT = ROOT/'results'/'2단계모델'": f"OUT = Path(r'{OUT / '03_2단계모델'}')",
}, skip=('# 9.', '# 10.'))
print(f'03 완료 ({time.time() - t0:.0f}초)')
t0 = time.time()
ns4, log4 = run_nb(NB04, {
    "V4_PATH = find('38_최종baseline_V4.csv')": f"V4_PATH = Path(r'{V4_ORIG}')",
    "TYPE_PATH = find('유형나누기_결과.xlsx')": f"TYPE_PATH = Path(r'{TYPE_X}')",
    "OUT = ROOT/'results'/'감소분해'": f"OUT = Path(r'{OUT / '04_감소분해'}')",
}, stop='# 6.')
print(f'04 완료 ({time.time() - t0:.0f}초)')
PE, MAE, AXV, REC, SUMM = ns3['PE'], ns3['MAE'], ns3['AXV'], ns3['REC'], ns3['SUMM']
T = ns4['T']
assert len(T) == 43


## 3. 발표 그래프

In [ ]:
# G1. 12쪽 왼쪽 — 평시 범위 히스토그램 (판정에 실제로 쓴 1,000번 추출 그대로)
row = EPI[(EPI['gkey'] == EX_G) & (EPI['에피소드'] == EX_EP)].iloc[0]
key = [k for k in ns2['_CAP'] if k[0] == EX_G and set(k[1]) <= set(EP2.loc[(EP2['대상'] == row['대상']) & (EP2['에피소드'] == EX_EP), '날짜'].iloc[0])]
draws = ns2['_CAP'][key[0]] * 100
lo, hi, rate = row['평시하한'] * 100, row['평시상한'] * 100, row['차이율'] * 100
assert abs(np.percentile(draws, 5) - lo) < 1e-6
name = f"{EX_G.split('|')[0][:2]} {EX_G.split('|')[1]} · {EX_EP}"
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.axvspan(lo, hi, color=C['band'], zorder=0, label=f'평시 범위 5~95% ({lo:.1f}% ~ {hi:+.1f}%)')
ax.hist(draws, bins=40, color=C['blue'], alpha=.85, zorder=2)
ax.axvline(lo, color=C['teal'], ls='--', lw=1.5, zorder=3, label=f'판정 하한 {lo:.1f}%')
ax.axvline(rate, color=C['red'], lw=3, zorder=4, label=f'실제 {EX_EP} {rate:.1f}% → 감소')
ax.set_xlabel('기간 매출 차이율 (%) = (실제 합 - 평소 합) / 평소 합'); ax.set_ylabel('추출 횟수')
ax.set_title(f'{name}: 같은 날짜 수·요일 구성의 평시 기간을 1,000번 뽑아 만든 범위', loc='left', fontsize=11, color=C['ink'])
ax.legend(frameon=False, fontsize=9, loc='upper right')
save(fig, 'G1_평시범위_히스토그램.png')


In [ ]:
# G2. 12쪽 오른쪽 — 3축 예시 (하루 변화율, 3일 평균, 누적 면적, 회복)
e = EP2[(EP2['대상'] == row['대상']) & (EP2['에피소드'] == EX_EP)].iloc[0]
a, b = e['시작'] - pd.Timedelta(days=4), e['종료'] + pd.Timedelta(days=10)
y = D2[D2['gkey'] == EX_G].set_index('날짜')['y_day'].reindex(pd.date_range(a, b)) * 100
y[y.index.isin(ns2['SPECIAL2_DAYS'])] = np.nan
ev_days = pd.date_range(e['시작'], e['종료'])
ma = y.reindex(ev_days).rolling(3, min_periods=3).mean()
imp_day = ma.idxmin()
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.axvspan(e['시작'], e['종료'] + pd.Timedelta(days=1), color=C['shade'], zorder=0, lw=0, label='특보 기간')
yy = y.reindex(ev_days)
ax.fill_between(ev_days, yy.clip(upper=0), 0, color=C['orange'], alpha=.35, step=None, zorder=1, label=f"누적 손실 = 평소 하루 매출 {abs(row['Cumulative']):.1f}일치")
ax.plot(y.index, y.values, color=C['gray'], lw=1.2, zorder=2, label='하루 변화율')
ax.plot(ma.index, ma.values, color=C['navy'], lw=2.4, zorder=3, label='3일 평균')
ax.scatter([imp_day], [ma.min()], color=C['red'], s=60, zorder=4)
ax.annotate(f"Impact {row['Impact']*100:.1f}%\n(3일 평균 최저)", (imp_day, ma.min()), xytext=(-12, -4), textcoords='offset points', ha='right', va='top', color=C['red'], fontsize=10)
ax.axvline(e['종료'] + pd.Timedelta(days=1), color=C['ink2'], ls=':', lw=1)
rec = row['회복']; rec_txt = '정보 없음' if not np.isfinite(rec) else f"{rec:.0f}일" + (' 이상' if row['이상'] else '')
ax.text(e['종료'] + pd.Timedelta(days=1.3), ax.get_ylim()[1] * 0.9 if ax.get_ylim()[1] > 0 else 2, f'종료 후 회복: {rec_txt}', fontsize=9, color=C['ink2'], va='top')
ax.axhline(0, color=C['ink2'], lw=1)
ax.set_ylabel('평소 대비 매출 변화율 (%)')
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%m/%d'))
ax.set_title(f'{name}: 깊이·누적·회복은 서로 다른 피해를 보여준다', loc='left', fontsize=11, color=C['ink'])
ax.set_ylim(min(ax.get_ylim()[0], ma.min() - 12), ax.get_ylim()[1])
ax.legend(frameon=False, fontsize=9, loc='upper center', bbox_to_anchor=(.5, -0.1), ncol=4)
save(fig, 'G2_3축_예시.png')
print(f"[12쪽 숫자] 차이율 {rate:.1f}%, 판정 하한 {lo:.1f}%, Impact {row['Impact']*100:.1f}%, 누적 {row['Cumulative']:.2f}일치, 회복 {rec_txt}")


In [ ]:
# G3·G4. 13쪽 — 이벤트별 업종 순위상관 / 칸 단위 MAE 개선율
ORDER = [('강남(서울동남권)', '폭염'), ('강남(서울동남권)', '호우'), ('춘천', '폭염'), ('춘천', '호우')]
lab = lambda r, t: f'{r[:2]} {t}'
fig, axes = plt.subplots(1, 4, figsize=(13, 3.9), sharey=True, gridspec_kw=dict(width_ratios=[4, 5, 4, 6]))
for ax, (reg, typ) in zip(axes, ORDER):
    z = PE[(PE['대상'] == reg) & (PE['종류'] == typ)].reset_index(drop=True); x = np.arange(len(z))
    for j, r in z.iterrows(): ax.plot([j, j], [r['과거평균'], r['모델']], color='#c9ccd3', lw=2, zorder=1)
    ax.scatter(x - .08, z['과거평균'], s=55, color=C['orange'], edgecolor='white', lw=1.2, zorder=3, label='과거 평균 예측')
    ax.scatter(x + .08, z['모델'], s=55, color=C['blue'], edgecolor='white', lw=1.2, zorder=3, label='기상 모델')
    s = SUMM[(SUMM['대상'] == reg) & (SUMM['종류'] == typ)].iloc[0]
    ax.set_title(f"{lab(reg, typ)}  (평균 {s['모델_평균']:.2f} vs {s['과거평균_평균']:.2f}, 모델 {s['모델승']})", fontsize=9.5, color=C['ink'], loc='left')
    ax.set_xticks(x, z['에피소드'], fontsize=8.5); ax.axhline(0, color=C['ink2'], lw=1); ax.set_xlim(-.5, len(z) - .5)
axes[0].set_ylabel('업종 순위상관 (실제 vs 예측)')
h, l = axes[0].get_legend_handles_labels(); fig.legend(h, l, frameon=False, ncol=2, loc='upper right', bbox_to_anchor=(1, 1.06))
fig.suptitle('이벤트마다 업종 피해 순위를 얼마나 맞혔나 (그 이벤트를 빼고 학습한 예측)', x=0.01, ha='left', y=1.06, fontsize=11, color=C['ink'])
fig.tight_layout(); save(fig, 'G3_이벤트별_순위상관.png')

M = MAE.set_index(['대상', '종류']).loc[ORDER]
fig, ax = plt.subplots(figsize=(7.5, 3.9)); x = np.arange(4); w = .36
for off, col, color, nm in [(-w/2, '과거평균 개선율(평소대로 대비)', C['gray'], '과거 평균 예측'), (w/2, '모델 개선율(평소대로 대비)', C['blue'], '기상 모델')]:
    v = M[col].to_numpy() * 100
    bars = ax.bar(x + off, v, w, color=color, label=nm)
    for bx, vv in zip(x + off, v): ax.text(bx, vv + (0.4 if vv >= 0 else -0.4), f'{vv:+.1f}%', ha='center', va='bottom' if vv >= 0 else 'top', fontsize=9)
ax.axhline(0, color=C['ink2'], lw=1); ax.set_xticks(x, [lab(r, t) for r, t in ORDER])
ax.set_ylabel("'평소대로(y=0)' 대비 MAE 개선율 (%)"); ax.legend(frameon=False, fontsize=9, loc='lower left')
ax.set_title('특보일 칸 단위 y 오차: 모델은 줄이고, 과거 평균은 오히려 늘림', loc='left', fontsize=11, color=C['ink'])
ax.set_ylim(min(-16, ax.get_ylim()[0]), max(10, ax.get_ylim()[1]))
save(fig, 'G4_MAE_개선율.png')


In [ ]:
# G5. 15쪽 — Impact × Cumulative 점수 (후보끼리 백분위, 1 = 가장 취약)
SHAPE_C = {'심각형': C['red'], '급락형': C['orange'], '지속 침체형': C['yellow'], '상대적 완만형': C['blue']}
fig, axes = plt.subplots(2, 2, figsize=(11, 9.2))
for ax, (reg, typ) in zip(axes.ravel(), ORDER):
    g = CAND[(CAND['대상'] == reg) & (CAND['종류'] == typ)]
    ax.add_patch(plt.Rectangle((0.5, 0.5), 0.62, 0.62, color=C['shade'], zorder=0, lw=0))
    ax.axvline(0.5, color=C['ink2'], lw=1, ls='--'); ax.axhline(0.5, color=C['ink2'], lw=1, ls='--')
    for t, c in SHAPE_C.items():
        q = g[g['피해모양'] == t]
        if len(q): ax.scatter(q['I점수'], q['C점수'], s=60, color=c, edgecolor='white', lw=1.3, zorder=3, label='상대적 심각형' if t == '심각형' else t)
    for x_, y_, t in [(.99, .99, '상대적 심각형'), (.99, .01, '급락형'), (.01, .99, '지속 침체형'), (.01, .01, '상대적 완만형')]:
        ax.text(x_, y_, t, transform=ax.transAxes, ha='right' if x_ > .5 else 'left', va='top' if y_ > .5 else 'bottom', fontsize=8.5, color=C['ink2'])
    top = g[g['순위'] <= 3]
    for _, r in top.iterrows():
        right = r['I점수'] > 0.7
        ax.annotate(f"{int(r['순위'])}위 {r['업종']}", (r['I점수'], r['C점수']), xytext=(-7 if right else 7, -3), textcoords='offset points',
                    ha='right' if right else 'left', va='center', fontsize=8.5, color=C['ink'])
    ax.set_xlim(0, 1.12); ax.set_ylim(0, 1.12)
    ax.set_xlabel('Impact 점수 (깊이)'); ax.set_ylabel('Cumulative 점수 (누적)')
    ax.set_title(f'{lab(reg, typ)} — 후보 {len(g)}개 (이름: 3위 이내)', loc='left', fontsize=10.5, color=C['ink'])
h, l = [], []
for a_ in axes.ravel():
    for hh, ll in zip(*a_.get_legend_handles_labels()):
        if ll not in l: h.append(hh); l.append(ll)
fig.legend(h, l, frameon=False, ncol=4, loc='upper center', bbox_to_anchor=(.5, 1.02))
fig.tight_layout(); save(fig, 'G5_Impact_Cumulative.png')


In [ ]:
# G6. 16쪽 왼쪽 — 방문 vs 지출 (주 결과: 평시 객단가 기준) / G6b 보조: 여름 객단가 기준
VS_C = {'방문 감소형': C['blue'], '혼합형': C['teal'], '지출 감소형': C['orange'], '감소 아님': C['gray']}
def vs_plot(sfx, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    for ax, typ in zip(axes, ['폭염', '호우']):
        z = T[T['종류'] == typ]
        ax.axhline(0, color=C['ink2'], lw=1); ax.axvline(0, color=C['ink2'], lw=1)
        for t, c in VS_C.items():
            q = z[z[f'분해_유형{sfx}'] == t]
            if len(q): ax.scatter(q[f'분해_건수변화{sfx}'] * 100, q[f'분해_객단가변화{sfx}'] * 100, s=55, color=c, edgecolor='white', lw=1.2, zorder=3, label=f'{t} {len(q)}')
        for j, (_, r) in enumerate(z[z['순위'] == 1].sort_values(f'분해_건수변화{sfx}').iterrows()):
            ax.annotate(f"{r['대상'][:2]} {r['업종']}", (r[f'분해_건수변화{sfx}'] * 100, r[f'분해_객단가변화{sfx}'] * 100), xytext=(6, 6 if j % 2 == 0 else -14),
                        textcoords='offset points', fontsize=8.5, color=C['ink'])
        ax.margins(x=0.08, y=0.12)
        ax.set_xlabel('평소 대비 결제건수 변화 (%)'); ax.set_ylabel('평소 대비 객단가 변화 (%)')
        ax.set_title(f'{typ} (후보 {len(z)}개, 이름: 1위)', loc='left', fontsize=10.5, color=C['ink']); ax.legend(frameon=False, fontsize=9, loc='upper center', bbox_to_anchor=(.5, -0.16), ncol=4)
    fig.suptitle(title, x=0.01, ha='left', fontsize=11, color=C['ink']); fig.tight_layout(); save(fig, fname)
vs_plot('', '방문(건수) vs 지출(객단가) — 주 결과: 평시 객단가 기준', 'G6_방문_지출.png')
vs_plot('_여름', '보조: 여름(8월~9월 상순) 객단가 기준', 'G6b_방문_지출_여름보조.png')


In [ ]:
# G7. 16쪽 오른쪽 — 고객층(외지인/거주민) 판정
CUST = [('거주민 주도', C['blue']), ('쏠림 없음', C['teal']), ('외지인 주도', C['gray'])]
X = pd.crosstab([T['대상'], T['종류']], T['고객층']).reindex(ORDER).fillna(0)
fig, ax = plt.subplots(figsize=(8, 3.6)); left = np.zeros(len(X)); yy = np.arange(len(X))[::-1]
for k, c in CUST:
    v = X[k].to_numpy() if k in X else np.zeros(len(X))
    ax.barh(yy, v, left=left, color=c, label=k, height=.6)
    for yv, lv, vv in zip(yy, left, v):
        if vv: ax.text(lv + vv / 2, yv, f'{int(vv)}', ha='center', va='center', color='white', fontsize=10, fontweight='bold')
    left += v
ax.set_yticks(yy, [lab(r, t) for r, t in ORDER]); ax.set_xlabel('취약 후보 수 (개)')
ax.legend(frameon=False, ncol=3, fontsize=9, loc='lower right', bbox_to_anchor=(1, 1))
ax.set_title(f"고객층 쏠림: {int(X['쏠림 없음'].sum())} / {int(X.to_numpy().sum())}개 업종에서 특정 고객층 쏠림 없음", loc='left', fontsize=11, color=C['ink'], pad=24)
ax.grid(axis='y', visible=False)
save(fig, 'G7_고객층.png')
print('※ 강남의 거주민은 서울 시민 전체라 강남 고객층 결과는 참고용')


## 4. 슬라이드 숫자표

In [ ]:
# 3. 슬라이드에 넣을 숫자 — 화면에 출력하고 발표숫자.xlsx로 저장
pct = lambda v: f'{v*100:+.1f}%'
TAB = {}
# 14쪽: 순위 (공동 순위 표시)
top = CAND.sort_values(['대상', '종류', '순위', '업종'])[['대상', '종류', '순위', '업종', '반복', 'Impact', 'Cumulative', 'Recovery표시', '피해모양', '시간대유형']].copy()
top['Impact'] = (top['Impact'] * 100).round(1); top['Cumulative'] = top['Cumulative'].round(2); top['순위'] = top['순위'].astype(int)
TAB['14쪽_순위'] = top
print('[14쪽] 지역·기상별 상위 5 (같은 숫자 = 공동 순위)')
for (r, t), z in top.groupby(['대상', '종류'], sort=False):
    print(f'  {lab(r, t)} ({len(z)}개): ' + ', '.join(f"{q.순위}위 {q.업종}" for q in z[z['순위'] <= 5].itertuples()))
# 15쪽: 피해 모양·회복 지연
shp = pd.crosstab([CAND['대상'], CAND['종류']], CAND['피해모양']).reindex(ORDER)
shp['회복지연(3일 이상)'] = CAND.assign(f=CAND['Recovery'] >= 3).groupby(['대상', '종류'])['f'].sum().reindex(ORDER)
TAB['15쪽_피해모양'] = shp.reset_index(); print('\n[15쪽] 피해 모양'); print(shp.to_string())
# 16쪽: 시간대·방문/지출·고객층
tt = pd.crosstab([CAND['대상'], CAND['종류']], CAND['시간대유형']).reindex(ORDER).fillna(0).astype(int)
vs = pd.crosstab([T['대상'], T['종류']], T['분해_유형']).reindex(ORDER).fillna(0).astype(int)
vs2 = pd.crosstab([T['대상'], T['종류']], T['분해_유형_여름']).reindex(ORDER).fillna(0).astype(int)
cu = pd.crosstab([T['대상'], T['종류']], T['고객층']).reindex(ORDER).fillna(0).astype(int)
TAB['16쪽_시간대'], TAB['16쪽_방문지출_주'], TAB['16쪽_방문지출_여름보조'], TAB['16쪽_고객층'] = tt.reset_index(), vs.reset_index(), vs2.reset_index(), cu.reset_index()
print('\n[16쪽] 시간대 유형'); print(tt.to_string())
print('\n[16쪽] 방문/지출 — 주 결과'); print(vs.to_string(), '\n  합계:', vs.sum().to_dict())
print('\n[16쪽] 방문/지출 — 보조(여름 객단가)'); print(vs2.to_string(), '\n  합계:', vs2.sum().to_dict())
print('\n[16쪽] 고객층'); print(cu.to_string(), '\n  합계:', cu.sum().to_dict())
cr = tt.loc[('춘천', '호우')]
print(f"\n  KPI 예시 — WHEN: 춘천 호우 전일 감소형 {cr.get('전일 감소형', 0)}/{cr.sum()}, HOW: 방문 감소형 주 {vs.sum().get('방문 감소형', 0)}/43 · 여름 보조 {vs2.sum().get('방문 감소형', 0)}/43, WHO: 쏠림 없음 {cu.sum().get('쏠림 없음', 0)}/43")
# 13쪽: 모델 vs 과거 평균
view = SUMM.merge(MAE[['대상', '종류', '모델 개선율(평소대로 대비)', '과거평균 개선율(평소대로 대비)']], on=['대상', '종류'])
imp = AXV[AXV['축'] == 'Impact'][['대상', '종류', '모델', '과거 평균']].rename(columns={'모델': 'Impact순위상관_모델', '과거 평균': 'Impact순위상관_과거평균'})
view = view.merge(imp, on=['대상', '종류']).merge(REC[['대상', '종류', '실제후보', '모델_재현율', '과거 평균_재현율', '모델_상위겹침', '과거 평균_상위겹침', '상위k']], on=['대상', '종류'])
TAB['13쪽_모델비교'] = view; print('\n[13쪽] 모델 vs 과거 평균'); print(view.round(3).to_string(index=False))
print(f"  KPI 예시 — MAE 개선 {view['모델 개선율(평소대로 대비)'].min()*100:.1f}~{view['모델 개선율(평소대로 대비)'].max()*100:.1f}%, "
      f"Impact 순위상관 모델 {view['Impact순위상관_모델'].min():.2f}~{view['Impact순위상관_모델'].max():.2f} (과거 평균 {view['Impact순위상관_과거평균'].min():.2f}~{view['Impact순위상관_과거평균'].max():.2f}), "
      f"사전 규칙상 모델 우세 {int((view['판단'] == '모델이 업종 순위를 더 잘 맞힘').sum())}/4")
# 12쪽·보류
TAB['12쪽_예시'] = EPI[(EPI['gkey'] == EX_G) & (EPI['에피소드'] == EX_EP)]
TAB['분석범위보류'] = S2[S2['전체판정'] == '분석범위 보류']
print('\n[10·12쪽] 분석범위 보류:', TAB['분석범위보류'][['대상', '종류', '업종', '반복']].to_dict('records'))
print('전체 판정:', S2['전체판정'].value_counts().to_dict())
with pd.ExcelWriter(OUT / '발표숫자.xlsx') as xw:
    for k, v in TAB.items(): v.to_excel(xw, sheet_name=k[:31], index=False)
print('\n저장:', OUT / '발표숫자.xlsx')
